# 04 — Ruído, JPEG e FGSM no mesmo teste

FGSM não direcionado, caixa branca, com rótulo verdadeiro. A perturbação é calculada em RGB **[0,1] antes da normalização**. Epsilon 8/255 corresponde a até 8 níveis por canal. Nos labs anteriores, 8/255 era adicionado em [-1,1], equivalente a 4/255 nos pixels: os resultados históricos não são a mesma condição.

ASR será computada no notebook 05 apenas entre amostras inicialmente corretas. A acurácia adversarial usa todas as amostras.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/tcc_justica')
ROOT.mkdir(parents=True, exist_ok=True)
print('Pasta persistente:', ROOT)


In [ ]:
%pip -q install timm==1.0.22 scikit-learn pandas matplotlib scipy pillow


In [ ]:
import os, random, json, hashlib, io
import numpy as np, pandas as pd
import torch, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARCH='legacy_xception'
SIZE=299
class Faces(Dataset):
    def __init__(self,df,augment=False):
        self.df=df.reset_index(drop=True)
        self.tf=transforms.Compose([transforms.Resize((SIZE,SIZE))]+([transforms.RandomHorizontalFlip()] if augment else [])+[transforms.ToTensor()])
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.image_path) as im:x=self.tf(im.convert('RGB'))
        return x,int(r.label),i

def normalized(x):return (x-.5)/.5

def load_trained(path):
    assert Path(path).is_file(), 'Falta checkpoint treinado do notebook 02.'
    ck=torch.load(path,map_location='cpu',weights_only=True)
    assert ck.get('class_map')=={'real':0,'fake':1}, 'Checkpoint incompatível; use notebook 02.'
    assert ck['mean']==[.5]*3 and ck['std']==[.5]*3 and ck['size']==299
    model=timm.create_model(ck['arch'],pretrained=False,num_classes=2)
    model.load_state_dict(ck['state_dict'],strict=True)
    return model.to(DEVICE).eval(),ck

def check_manifest(df):
    assert df.image_id.is_unique
    assert set(df.label)=={0,1} and set(df.split)=={'train','val','test'}
    assert df.groupby('unit_id').split.nunique().max()==1
    assert df.groupby('pair_id').split.nunique().max()==1
    owners={}
    for r in df.itertuples():
        for u in [r.unit_id,r.source_unit_id]:
            assert owners.setdefault(u,r.split)==r.split, 'Fonte/alvo cruzam splits.'
    assert df.image_path.map(lambda p:Path(p).is_file()).all(), 'Há imagens ausentes.'

def manifest_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()


In [ ]:
DATASET=ROOT/'geracoes'/'piloto_10'/'dataset.csv'
CHECKPOINT=ROOT/'modelos'/'xception_piloto_10'/'detector.pt'
RESULT=ROOT/'resultados'/'xception_piloto_10';RESULT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATASET);check_manifest(df)
model,ck=load_trained(CHECKPOINT)
assert manifest_hash(DATASET)==ck['manifest_sha256'], 'Dataset diferente daquele associado ao checkpoint.'
test=df[df.split=='test'].reset_index(drop=True)
assert len(test)>0
loader=DataLoader(Faces(test),batch_size=16,shuffle=False,num_workers=2)
print('Teste:',len(test),'imagens')


In [ ]:
EPSILONS=[0.,2/255,4/255,8/255]
SIGMA=15/255;JPEG_QUALITY=30
rng=torch.Generator(device=DEVICE).manual_seed(SEED)
rows=[]
def record(indices,condition,probs):
    for idx,p in zip(indices.tolist(),probs.detach().cpu().tolist()):
        r=test.iloc[idx].to_dict();r.update(condition=condition,p_fake=p,pred=int(p>=.5));rows.append(r)
def jpeg_batch(x):
    imgs=[]
    for sample in x:
        im=transforms.ToPILImage()(sample.cpu());buf=io.BytesIO()
        im.save(buf,format='JPEG',quality=JPEG_QUALITY);buf.seek(0)
        with Image.open(buf) as decoded:imgs.append(transforms.ToTensor()(decoded.convert('RGB')))
    return torch.stack(imgs).to(DEVICE)
for x,y,indices in loader:
    x,y=x.to(DEVICE),y.to(DEVICE)
    with torch.no_grad():
        record(indices,'clean',model(normalized(x)).softmax(1)[:,1])
        noisy=(x+SIGMA*torch.randn(x.shape,device=DEVICE,generator=rng)).clamp(0,1)
        record(indices,'noise_15',model(normalized(noisy)).softmax(1)[:,1])
        record(indices,'jpeg_30',model(normalized(jpeg_batch(x))).softmax(1)[:,1])
    xgrad=x.detach().clone().requires_grad_(True)
    loss=nn.functional.cross_entropy(model(normalized(xgrad)),y)
    grad=torch.autograd.grad(loss,xgrad)[0]
    for eps in EPSILONS:
        adv=(x+eps*grad.sign()).clamp(0,1).detach()
        assert (adv-x).abs().max().item()<=eps+1e-6
        with torch.no_grad():
            record(indices,f'fgsm_{round(eps*255)}',model(normalized(adv)).softmax(1)[:,1])
results=pd.DataFrame(rows)
clean=results[results.condition=='clean'].set_index('image_id').pred
zero=results[results.condition=='fgsm_0'].set_index('image_id').pred
assert clean.equals(zero), 'FGSM epsilon zero deve reproduzir predição limpa.'
results.to_csv(RESULT/'predicoes_robustez.csv',index=False)
(RESULT/'config_robustez.json').write_text(json.dumps({'epsilons':EPSILONS,'sigma':SIGMA,
    'jpeg_quality':JPEG_QUALITY,'seed':SEED,'checkpoint_sha256':manifest_hash(CHECKPOINT)},indent=2))
print('Concluído:',RESULT/'predicoes_robustez.csv')


Referência de implementação: https://docs.pytorch.org/tutorials/beginner/fgsm_tutorial.html . A ameaça modelada tem acesso aos pesos e gradientes; não equivale a todas as condições de uso real.
